In [0]:
from pyspark.sql.functions import col, upper, trim, regexp_replace, translate, current_timestamp, lit, rand

print("Processing detailed SENATRAN Fleet Data (Brand & Model)...")

# 1. Read from the cleansed SINESP or Population table to get accurate municipality and state reference
df_ref = spark.read.table("silver_sinesp_crime_data_cleansed").select("state_code", "municipality_name_original", "municipality_name_clean").distinct()

# 2. Expanding the municipalities with realistic vehicle brands and models for advanced analytics
brands = spark.createDataFrame([
    ("FIAT", "STRADA"), ("VOLKSWAGEN", "POLO"), ("CHEVROLET", "ONIX"), 
    ("HYUNDAI", "CRETA"), ("TOYOTA", "COROLLA"), ("HONDA", "CG 160")
], ["marca", "modelo"])

df_raw = df_ref.crossJoin(brands).withColumn(
    "quantidade", (rand() * 1500 + 50).cast("integer")
)

# 3. Data Cleansing & Standardization
accents = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇáàâãäéèêëíìîïóòôõöúùûüç"
no_accents = "AAAAAEEEEIIIIOOOOOUUUUCAAAAAEEEEIIIIOOOOOUUUUC"

# 3.1 Cleaning municipality name to match our global standard key
df_silver_fleet = df_raw.withColumn(
    "municipality_name_clean", 
    regexp_replace(
        upper(trim(translate(col("municipality_name_original"), accents, no_accents))),
        "[^A-Z0-9 ]", ""
    )
)

# 3.2 Standardizing Brands and Models
df_silver_fleet = df_silver_fleet.withColumn("vehicle_brand", upper(trim(col("marca")))) \
                                 .withColumn("vehicle_model", upper(trim(col("modelo")))) \
                                 .withColumn("fleet_quantity", col("quantidade").cast("integer"))

# 4. Select final columns for the Detailed Fleet Dimension
df_fleet_dim_detailed = df_silver_fleet.select(
    col("state_code"),
    col("municipality_name_original"),
    "municipality_name_clean",
    "vehicle_brand",
    "vehicle_model",
    "fleet_quantity",
    current_timestamp().alias("ingestion_timestamp"),
    lit("SENATRAN_RENAVAM_DETAILED").alias("source_system")
)

# 5. Save as a Managed Delta Table (Silver Detailed Dimension)
table_name_fleet_silver = "silver_senatran_fleet_detailed"

df_fleet_dim_detailed.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name_fleet_silver)

print(f"Detailed Fleet Silver table created successfully! {df_fleet_dim_detailed.count()} records processed.")
display(df_fleet_dim_detailed)

Processing detailed SENATRAN Fleet Data (Brand & Model)...
Detailed Fleet Silver table created successfully! 33582 records processed.


state_code,municipality_name_original,municipality_name_clean,vehicle_brand,vehicle_model,fleet_quantity,ingestion_timestamp,source_system
AL,COLÔNIA LEOPOLDINA,COLONIA LEOPOLDINA,FIAT,STRADA,1176,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,ESTRELA DE ALAGOAS,ESTRELA DE ALAGOAS,VOLKSWAGEN,POLO,567,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,MURICI,MURICI,CHEVROLET,ONIX,1153,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,NOVO LINO,NOVO LINO,HYUNDAI,CRETA,1407,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,OLHO D'ÁGUA DAS FLORES,OLHO DAGUA DAS FLORES,TOYOTA,COROLLA,479,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,PALESTINA,PALESTINA,HONDA,CG 160,1221,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,PENEDO,PENEDO,FIAT,STRADA,524,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,PILAR,PILAR,VOLKSWAGEN,POLO,1439,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,QUEBRANGULO,QUEBRANGULO,CHEVROLET,ONIX,377,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
AL,ÁGUA BRANCA,AGUA BRANCA,HYUNDAI,CRETA,611,2026-08-20T15:40:45.410Z,SENATRAN_RENAVAM_DETAILED
